# Day 34 AM – SVM & KNN: Handwritten Digit Classifier

MNIST-style digit classification using `load_digits`, SVM (RBF + GridSearchCV), KNN (optimal K), confusion analysis, and FAISS stretch task.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)


## Part A – Digit Classifier: SVM and KNN

In [ ]:
digits = load_digits()
X = digits.data
y = digits.target

print("Shape:", X.shape)
print("Classes:", np.unique(y))
print("Samples per class:", np.bincount(y))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)


In [ ]:
param_grid = {
    "C":     [0.1, 1, 10, 50],
    "gamma": [0.001, 0.005, 0.01, "scale"]
}

grid = GridSearchCV(SVC(kernel="rbf"), param_grid, cv=5,
                    scoring="accuracy", n_jobs=-1)
grid.fit(X_train_sc, y_train)

svm_best = grid.best_estimator_
y_pred_svm = svm_best.predict(X_test_sc)

print("Best params:", grid.best_params_)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm).round(4))


In [ ]:
k_scores = {}
for k in range(1, 16):
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_sc, y_train, cv=5, scoring="accuracy")
    k_scores[k] = scores.mean()

best_k = max(k_scores, key=k_scores.get)
print("Best K:", best_k, "| CV Accuracy:", round(k_scores[best_k], 4))

knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_sc, y_train)
y_pred_knn = knn_best.predict(X_test_sc)
print("KNN Test Accuracy:", accuracy_score(y_test, y_pred_knn).round(4))


In [ ]:
print("Classification Report – SVM")
print(classification_report(y_test, y_pred_svm))

print("Classification Report – KNN")
print(classification_report(y_test, y_pred_knn))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_svm, ax=axes[0], colorbar=False)
axes[0].set_title("SVM (RBF) Confusion Matrix")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_knn, ax=axes[1], colorbar=False)
axes[1].set_title(f"KNN (K={best_k}) Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()


In [ ]:
cm = confusion_matrix(y_test, y_pred_svm)
np.fill_diagonal(cm, 0)

confused_pairs = []
for i in range(10):
    for j in range(i+1, 10):
        errors = cm[i, j] + cm[j, i]
        if errors > 0:
            confused_pairs.append((errors, i, j))

confused_pairs.sort(reverse=True)
print("Most confused digit pairs (SVM):")
for count, a, b in confused_pairs[:5]:
    print(f"  Digit {a} vs Digit {b}: {count} errors")


## Part B – Stretch: FAISS Approximate Nearest Neighbors

In [ ]:
import time

try:
    import faiss

    X_train_f = X_train_sc.astype("float32")
    X_test_f  = X_test_sc.astype("float32")

    d = X_train_f.shape[1]
    index = faiss.IndexFlatL2(d)
    index.add(X_train_f)

    query = X_test_f[:1000]

    start = time.time()
    D, I = index.search(query, best_k)
    faiss_time = time.time() - start

    faiss_preds = []
    for neighbors in I:
        neighbor_labels = y_train[neighbors]
        pred = np.bincount(neighbor_labels).argmax()
        faiss_preds.append(pred)
    faiss_preds = np.array(faiss_preds)

    start = time.time()
    knn_preds_1000 = knn_best.predict(X_test_sc[:1000])
    sklearn_time = time.time() - start

    print("FAISS Accuracy (1000 queries) :", accuracy_score(y_test[:1000], faiss_preds).round(4))
    print("sklearn KNN Accuracy (1000)   :", accuracy_score(y_test[:1000], knn_preds_1000).round(4))
    print(f"FAISS time   : {faiss_time*1000:.2f} ms")
    print(f"sklearn time : {sklearn_time*1000:.2f} ms")

except ImportError:
    print("FAISS not installed. Run: pip install faiss-cpu")
    print("See faiss_comparison.md for documented findings.")
